In [ ]:
import pandas as pd
import numpy as np
import requests
import os
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Action Plan API (local)
API_BASE_URL = "https://localhost:7204/api"
ENDPOINT = f"{API_BASE_URL}/ml-features/training-dataset"

records = []
data_source = "api"

try:
    # Jeton admin ou cle dataset (MlFeatures:DatasetApiKey cote API)
    headers = {"X-Dataset-Key": os.environ.get("COFICAB_DATASET_KEY", "")}
    response = requests.get(ENDPOINT, headers=headers, verify=False, timeout=60)
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, list):
        records = payload
    elif isinstance(payload, dict):
        records = payload.get("items") or payload.get("data") or payload.get("rows") or []
    else:
        raise ValueError(f"Format de réponse inattendu: {type(payload)}")
    if not records:
        raise ValueError("Dataset API vide.")
except Exception as exc:
    # Fallback local : l'endpoint /api/ml-features/training-dataset n'est pas encore
    # disponible (404). Seed "réel" minimal pour débloquer le pipeline.
    data_source = "fallback_seed"
    print(f"WARNING: chargement API impossible ({exc}). Utilisation d'un seed local.")
    rng = np.random.default_rng(7)
    defects = ["Soudure défectueuse", "Isolation endommagée", "Dimension hors tolérance", "Coupure fil", "Non-conformité emballage"]
    severities = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
    depts = ["QUALITE", "PRODUCTION", "MAINTENANCE", "LOGISTIQUE"]
    n = 120
    skills = rng.integers(1, 6, size=n)
    loads = rng.uniform(0.15, 0.95, size=n)
    sev = rng.choice(severities, size=n, p=[0.25, 0.35, 0.25, 0.15])
    hard = np.array([{"LOW": 0.05, "MEDIUM": 0.12, "HIGH": 0.22, "CRITICAL": 0.35}[s] for s in sev])
    logit = -0.35 + 2.4 * (skills / 5.0) - 2.1 * loads - 1.6 * hard
    p = 1 / (1 + np.exp(-logit))
    records = [
        {
            "DefectType": defects[i % len(defects)],
            "Severity": sev[i],
            "DepartmentId": depts[i % len(depts)],
            "EmployeeSkill": int(skills[i]),
            "Workload": float(round(loads[i], 3)),
            "IsOnTime": bool(rng.random() < p[i]),
        }
        for i in range(n)
    ]

df = pd.DataFrame(records)

print(f"Source données : {data_source}")
print(f"Nombre de lignes : {len(df)}")
print("\nColonnes disponibles :")
print(list(df.columns))
print("\nTypes de données :")
print(df.dtypes)
print("\nValeurs manquantes par colonne :")
print(df.isna().sum())

df.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid", context="notebook")

# Normalise les noms de colonnes (camelCase / PascalCase API)
col_map = {c.lower(): c for c in df.columns}

def col(*candidates: str) -> str | None:
    for name in candidates:
        if name.lower() in col_map:
            return col_map[name.lower()]
    return None

target_col = col("IsOnTime", "isOnTime", "is_on_time")
cat_candidates = {
    "DefectType": col("DefectType", "defectType", "defect_type"),
    "Severity": col("Severity", "severity"),
    "DepartmentId": col("DepartmentId", "departmentId", "department_id"),
}
cat_cols = {label: name for label, name in cat_candidates.items() if name is not None}

# ── 1. Distribution de la cible IsOnTime ──────────────────────────────────
if target_col is None:
    print("Colonne cible IsOnTime introuvable.")
else:
    counts = df[target_col].value_counts(dropna=False)
    pct = df[target_col].value_counts(normalize=True, dropna=False) * 100
    target_summary = pd.DataFrame({"count": counts, "percentage": pct.round(2)})
    print("Distribution IsOnTime :")
    print(target_summary)

    fig, ax = plt.subplots(figsize=(6, 4))
    order = counts.index.astype(str)
    sns.barplot(
        x=order,
        y=counts.values,
        hue=order,
        palette="Set2",
        legend=False,
        ax=ax,
    )
    for i, (n, p) in enumerate(zip(counts.values, pct.values)):
        ax.text(i, n, f"{n}\n({p:.1f}%)", ha="center", va="bottom", fontsize=9)
    ax.set_title("Distribution de IsOnTime")
    ax.set_xlabel("IsOnTime")
    ax.set_ylabel("Nombre de dossiers")
    plt.tight_layout()
    plt.show()

# ── 2. Distributions catégorielles ────────────────────────────────────────
if not cat_cols:
    print("Aucune feature catégorielle (DefectType, Severity, DepartmentId) trouvée.")
else:
    n = len(cat_cols)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), squeeze=False)
    for ax, (label, name) in zip(axes[0], cat_cols.items()):
        order = df[name].astype(str).value_counts().index
        sns.countplot(data=df.assign(**{name: df[name].astype(str)}), y=name, order=order, ax=ax, color="#2563eb")
        ax.set_title(f"Distribution — {label}")
        ax.set_xlabel("Count")
        ax.set_ylabel(label)
    plt.tight_layout()
    plt.show()

# ── 3. Heatmap de corrélation (features numériques) ────────────────────────
num_df = df.select_dtypes(include=[np.number]).copy()
# Inclure IsOnTime booléen/0-1 si présent
if target_col is not None and target_col not in num_df.columns:
    mapped = df[target_col].map({True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0})
    if mapped.notna().any():
        num_df[target_col] = pd.to_numeric(mapped, errors="coerce")

if num_df.shape[1] < 2:
    print("Pas assez de features numériques pour une heatmap de corrélation.")
else:
    corr = num_df.corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(max(6, 0.55 * corr.shape[1] + 2), max(5, 0.5 * corr.shape[0] + 2)))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True, ax=ax)
    ax.set_title("Heatmap de corrélation — features numériques")
    plt.tight_layout()
    plt.show()


## Méthodologie — données synthétiques (réalistes)

Objectif : **compléter** le dataset réel sans saturer le modèle avec une grille exhaustive trop régulière.

### Principes
1. **Compétence par type de défaut** — chaque profil employé a un vecteur de compétences `(DefectType → score 1–5)`, pas un unique score global. Pour une ligne donnée, `EmployeeSkill` = compétence **du domaine du défaut** (ex. un expert « Dimensionnel » peut être faible sur « Documentation »).
2. **Probabilité métier cohérente** — `P(IsOnTime)` augmente avec la compétence domaine et diminue avec la charge (`Workload`) et la sévérité.
3. **Bruit / exceptions (~6 %)** — après le tirage Bernoulli, on **inverse** aléatoirement ~6 % des labels pour simuler imprévus (arrêt machine, absences, sous-estimation du délai), tout en gardant un signal plus exploitable qu’à 12 %.
4. **Valeurs manquantes** — ~8 % de `NaN` indépendants sur `EmployeeSkill`, `Workload`, `DepartmentId` (et parfois `Severity`) pour forcer le pipeline d’imputation.
5. **Volume maîtrisé** — **800–1000** lignes synthétiques (plus le seed/API réel), au lieu d’une grille complète × répétitions (~3600).

### Procédure
- Tirer `N_PROFILES` profils employés (compétences par défaut + charge de base).
- Pour chaque ligne : tirer un défaut, une sévérité, un département, un profil ; poser `EmployeeSkill = skill[défaut]` et une charge bruitée.
- Calculer `p`, tirer `IsOnTime`, appliquer flip d’exception, injecter des manquants.
- Fusionner avec le réel (`source=real|synthetic`) dans `df_final`.

### Tuning
Après sélection du meilleur modèle (F1 test, params par défaut), un **GridSearchCV 5-fold** affine ses hyperparamètres sur le train, puis on réévalue sur le test et on compare à la version non tunée.


In [ ]:
# ── Génération synthétique réaliste (voir markdown méthodologie) ───────────
rng = np.random.default_rng(42)

def resolve_col(*names: str) -> str | None:
    mapping = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in mapping:
            return mapping[n.lower()]
    return None

c_defect = resolve_col("DefectType", "defectType", "defect_type")
c_severity = resolve_col("Severity", "severity")
c_dept = resolve_col("DepartmentId", "departmentId", "department_id")
c_skill = resolve_col("EmployeeSkill", "employeeSkill", "SkillLevel", "skillLevel")
c_load = resolve_col("Workload", "workload", "Charge", "charge")
c_target = resolve_col("IsOnTime", "isOnTime", "is_on_time")

defect_levels = (
    sorted(df[c_defect].dropna().astype(str).unique().tolist())
    if c_defect
    else ["Dimensionnel", "Esthétique", "Fonctionnel", "Emballage", "Documentation"]
)
severity_levels = (
    sorted(df[c_severity].dropna().astype(str).unique().tolist())
    if c_severity
    else ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
)
dept_levels = (
    sorted(df[c_dept].dropna().astype(str).unique().tolist())[:8]
    if c_dept
    else ["QUALITE", "PRODUCTION", "MAINTENANCE", "LOGISTIQUE"]
)

severity_hardness = {
    "LOW": 0.05, "MEDIUM": 0.12, "HIGH": 0.22, "CRITICAL": 0.35,
    "low": 0.05, "medium": 0.12, "high": 0.22, "critical": 0.35,
}

TARGET_SYNTH = int(rng.integers(800, 1001))  # 800–1000 lignes
N_PROFILES = 40
EXCEPTION_RATE = 0.06  # ~6 % de labels inversés
MISSING_RATE = 0.08

# Profils : compétence DISTINCTE par type de défaut (+ spécialité)
profiles = []
for pid in range(N_PROFILES):
    specialty = defect_levels[pid % len(defect_levels)]
    skills = {}
    for d in defect_levels:
        base = float(rng.integers(1, 6))
        if d == specialty:
            base = float(np.clip(base + rng.integers(1, 3), 1, 5))
        else:
            base = float(np.clip(base - rng.integers(0, 2), 1, 5))
        skills[d] = base
    profiles.append(
        {
            "profileId": f"P{pid:02d}",
            "skills": skills,
            "base_load": float(rng.uniform(0.2, 0.85)),
            "home_dept": dept_levels[pid % len(dept_levels)],
        }
    )


def on_time_probability(domain_skill: float, load: float, severity: str) -> float:
    s = float(np.clip(domain_skill / 5.0, 0, 1))
    w = float(np.clip(load if load <= 1.5 else load / 100.0, 0, 1))
    hard = severity_hardness.get(str(severity), 0.15)
    logit = -0.35 + 2.4 * s - 2.1 * w - 1.6 * hard
    return float(np.clip(1.0 / (1.0 + np.exp(-logit)), 0.05, 0.95))


rows = []
for _ in range(TARGET_SYNTH):
    defect = str(rng.choice(defect_levels))
    severity = str(rng.choice(severity_levels))
    profile = profiles[int(rng.integers(0, len(profiles)))]
    # Compétence = score DU TYPE DE DÉFAUT (pas un score global unique)
    skill = float(profile["skills"][defect])
    load = float(np.clip(profile["base_load"] + rng.normal(0, 0.12), 0.05, 0.98))
    dept = profile["home_dept"] if rng.random() < 0.7 else str(rng.choice(dept_levels))

    p = on_time_probability(skill, load, severity)
    p = float(np.clip(p + rng.normal(0, 0.05), 0.02, 0.98))
    is_on_time = bool(rng.random() < p)

    # Exceptions métier : ~6 % de labels incohérents
    if rng.random() < EXCEPTION_RATE:
        is_on_time = not is_on_time

    row = {
        "DefectType": defect,
        "Severity": severity,
        "DepartmentId": dept,
        "EmployeeSkill": skill,
        "Workload": round(load, 3),
        "IsOnTime": is_on_time,
        "source": "synthetic",
    }

    # Valeurs manquantes réalistes
    for col in ("EmployeeSkill", "Workload", "DepartmentId", "Severity"):
        if rng.random() < MISSING_RATE:
            row[col] = np.nan

    rows.append(row)

df_synth = pd.DataFrame(rows)

df_real = df.copy()
df_real["source"] = "real"
rename_real = {}
if c_defect and c_defect != "DefectType":
    rename_real[c_defect] = "DefectType"
if c_severity and c_severity != "Severity":
    rename_real[c_severity] = "Severity"
if c_dept and c_dept != "DepartmentId":
    rename_real[c_dept] = "DepartmentId"
if c_skill and c_skill != "EmployeeSkill":
    rename_real[c_skill] = "EmployeeSkill"
if c_load and c_load != "Workload":
    rename_real[c_load] = "Workload"
if c_target and c_target != "IsOnTime":
    rename_real[c_target] = "IsOnTime"
df_real = df_real.rename(columns=rename_real)

for c in ["DefectType", "Severity", "DepartmentId", "EmployeeSkill", "Workload", "IsOnTime", "source"]:
    if c not in df_real.columns:
        df_real[c] = np.nan

df_final = pd.concat([df_real, df_synth], ignore_index=True, sort=False)

print(f"Réel       : {len(df_real):,} lignes")
print(f"Synthétique: {len(df_synth):,} lignes (cible 800–1000)")
print(f"Final      : {len(df_final):,} lignes")
print(f"Exceptions (flip) configurées : {EXCEPTION_RATE:.0%}")
print(f"Taux manquants synth (par colonne clé) ≈ {MISSING_RATE:.0%}")
print("\nManquants synthétiques :")
print(df_synth[["EmployeeSkill", "Workload", "DepartmentId", "Severity"]].isna().mean().mul(100).round(1).astype(str) + " %")
print("\nRépartition source :")
print(df_final["source"].value_counts())
print("\nIsOnTime par source (% à temps, hors NaN label) :")
print(
    df_final.dropna(subset=["IsOnTime"])
    .groupby("source")["IsOnTime"]
    .apply(lambda s: round(100 * s.astype(bool).mean(), 2))
)

ex = profiles[0]
print("\nExemple profil", ex["profileId"], "- compétences par DefectType :")
print(ex["skills"])
display(df_synth.head())
display(df_final.head())



In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Dataset de travail : fusion real+synthetic si disponible, sinon df brut
work = df_final.copy() if "df_final" in dir() else df.copy()

# Harmonise IsOnTime
y_col = "IsOnTime" if "IsOnTime" in work.columns else next(
    (c for c in work.columns if c.lower() in {"isontime", "is_on_time"}), None
)
if y_col is None:
    raise ValueError("Colonne cible IsOnTime introuvable.")

y_raw = work[y_col]
y = y_raw.map({True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0})
y = pd.to_numeric(y, errors="coerce")
# Lignes sans label exploitable
valid_mask = y.notna()
work = work.loc[valid_mask].copy()
y = y.loc[valid_mask].astype(int)

# Features : on exclut la cible, les IDs techniques et la traçabilité
exclude = {
    y_col, "source", "id", "Id", "claimId", "ClaimId", "actionPlanId", "ActionPlanId",
    "createdAt", "CreatedAt", "updatedAt", "UpdatedAt", "completedAt", "CompletedAt",
    "title", "Title", "description", "Description",
}
feature_cols = [c for c in work.columns if c not in exclude]
X_raw = work[feature_cols].copy()

# Types
cat_cols = X_raw.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
num_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()

# bool déjà dans cat ; s'assurer qu'aucun overlap
cat_cols = [c for c in cat_cols if c not in num_cols]

print(f"Lignes retenues : {len(X_raw)}")
print(f"Features catégorielles ({len(cat_cols)}) : {cat_cols}")
print(f"Features numériques ({len(num_cols)}) : {num_cols}")

numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
)

X_matrix = preprocessor.fit_transform(X_raw)

# Noms de colonnes après OneHot
try:
    feature_names = preprocessor.get_feature_names_out()
except Exception:
    feature_names = [f"f_{i}" for i in range(X_matrix.shape[1])]

X = pd.DataFrame(X_matrix, columns=feature_names, index=X_raw.index)
y = y.loc[X.index]

print(f"\nX shape : {X.shape}  |  y shape : {y.shape}")
print("Distribution y (IsOnTime) :")
print(y.value_counts(normalize=True).rename({0: "False", 1: "True"}).mul(100).round(2).astype(str) + " %")
print("\nAperçu X :")
display(X.head())


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Train : {X_train.shape[0]} lignes  |  Test : {X_test.shape[0]} lignes")
print("Répartition y_train :", y_train.value_counts(normalize=True).round(3).to_dict())
print("Répartition y_test  :", y_test.value_counts(normalize=True).round(3).to_dict())

baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train, y_train)

y_pred_baseline = baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)

print(f"\nBaseline DummyClassifier (most_frequent)")
print(f"Accuracy test : {baseline_accuracy:.4f}  ({100 * baseline_accuracy:.2f} %)")
print("-> Seuil minimum a battre par les vrais modeles.")
print("\nClassification report :")
print(classification_report(y_test, y_pred_baseline, digits=3, zero_division=0))
print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred_baseline))


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.utils.class_weight import compute_sample_weight

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42,
    ),
    "RandomForestClassifier": RandomForestClassifier(
        class_weight="balanced",
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),
    # GradientBoostingClassifier n'a pas class_weight → sample_weight "balanced" par fold
    "GradientBoostingClassifier": GradientBoostingClassifier(
        random_state=42,
    ),
}

cv_results = []

for name, model in models.items():
    if name == "GradientBoostingClassifier":
        fold_scores = []
        for tr_idx, va_idx in cv.split(X_train, y_train):
            X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
            y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
            sw = compute_sample_weight(class_weight="balanced", y=y_tr)
            model.fit(X_tr, y_tr, sample_weight=sw)
            fold_scores.append(accuracy_score(y_va, model.predict(X_va)))
        scores = np.asarray(fold_scores)
    else:
        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="accuracy",
            n_jobs=-1,
        )

    mean_acc = scores.mean()
    std_acc = scores.std()
    cv_results.append(
        {
            "model": name,
            "cv_accuracy_mean": mean_acc,
            "cv_accuracy_std": std_acc,
            "fold_scores": scores,
        }
    )
    print(f"{name:28s}  accuracy = {mean_acc:.4f} ± {std_acc:.4f}  | folds={np.round(scores, 4)}")

cv_summary = pd.DataFrame(cv_results).drop(columns=["fold_scores"])
print("\nRésumé CV 5-fold (train) :")
display(cv_summary.sort_values("cv_accuracy_mean", ascending=False))

if "baseline_accuracy" in dir():
    print(f"\nBaseline DummyClassifier (test) à battre : {baseline_accuracy:.4f}")


In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Ré-entraîne chaque modèle sur tout le train, puis évalue sur le test (jamais vu)
eval_models = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42,
    ),
    "RandomForestClassifier": RandomForestClassifier(
        class_weight="balanced",
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoostingClassifier": GradientBoostingClassifier(
        random_state=42,
    ),
}

test_rows = []
fig_cm, axes_cm = plt.subplots(1, 3, figsize=(14, 4))
fig_roc, ax_roc = plt.subplots(figsize=(7, 5))

for i, (name, model) in enumerate(eval_models.items()):
    if name == "GradientBoostingClassifier":
        sw = compute_sample_weight(class_weight="balanced", y=y_train)
        model.fit(X_train, y_train, sample_weight=sw)
    else:
        model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_test, y_score)
    except ValueError:
        auc = np.nan

    test_rows.append(
        {
            "model": name,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "roc_auc": auc,
        }
    )

    print("=" * 60)
    print(name)
    print(f"accuracy={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  F1={f1:.4f}  ROC-AUC={auc:.4f}")
    print(classification_report(y_test, y_pred, digits=3, zero_division=0))
    print("Matrice de confusion :")
    print(confusion_matrix(y_test, y_pred))

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        display_labels=["Late (0)", "OnTime (1)"],
        cmap="Blues",
        ax=axes_cm[i],
        colorbar=False,
    )
    axes_cm[i].set_title(name)

    RocCurveDisplay.from_predictions(
        y_test,
        y_score,
        name=f"{name} (AUC={auc:.3f})",
        ax=ax_roc,
    )

fig_cm.suptitle("Matrices de confusion — test set", y=1.02)
fig_cm.tight_layout()
plt.show()

ax_roc.plot([0, 1], [0, 1], "k--", label="Aléatoire (AUC=0.5)")
ax_roc.set_title("Courbes ROC — test set")
ax_roc.legend(loc="lower right", fontsize=8)
fig_roc.tight_layout()
plt.show()

test_summary = pd.DataFrame(test_rows).sort_values("f1", ascending=False)
print("\nTableau récapitulatif (test set) :")
display(test_summary.style.format({
    "accuracy": "{:.4f}",
    "precision": "{:.4f}",
    "recall": "{:.4f}",
    "f1": "{:.4f}",
    "roc_auc": "{:.4f}",
}))

if "baseline_accuracy" in dir():
    best = test_summary.iloc[0]
    print(
        f"\nMeilleur modèle (F1) : {best['model']} "
        f"| accuracy={best['accuracy']:.4f} vs baseline={baseline_accuracy:.4f}"
    )


In [ ]:
# ── GridSearchCV 5-fold sur le meilleur modèle (params par défaut, cell précédente) ──
from sklearn.model_selection import GridSearchCV

# Référence "précédente" (bruit 12 %, LR non tunée) — pipeline_run_report antérieur
PREV_REF = {
    "exception_rate": 0.12,
    "model": "LogisticRegression",
    "accuracy": 0.6223404255319149,
    "precision": 0.574468085106383,
    "recall": 0.6352941176470588,
    "f1": 0.6033519553072626,
    "roc_auc": 0.681781838949172,
}

untuned_best_name = test_summary.sort_values("f1", ascending=False).iloc[0]["model"]
untuned_row = test_summary.loc[test_summary["model"] == untuned_best_name].iloc[0].to_dict()
print(f"Meilleur modèle (défaut) : {untuned_best_name}")
print(
    f"  F1={untuned_row['f1']:.4f}  acc={untuned_row['accuracy']:.4f}  "
    f"ROC-AUC={untuned_row['roc_auc']:.4f}"
)

# Grilles ciblées — focus sur LogisticRegression (meilleur actuel) ; fallback RF/GBC
param_grids = {
    "LogisticRegression": {
        "C": [0.01, 0.1, 1.0, 10.0],
        "solver": ["lbfgs", "liblinear"],
        "class_weight": ["balanced"],
        "max_iter": [2000],
        "random_state": [42],
    },
    "RandomForestClassifier": {
        "n_estimators": [100, 200, 300],
        "max_depth": [None, 8, 16],
        "min_samples_leaf": [1, 2, 4],
        "class_weight": ["balanced"],
        "random_state": [42],
        "n_jobs": [-1],
    },
    "GradientBoostingClassifier": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [2, 3],
        "random_state": [42],
    },
}

if untuned_best_name not in param_grids:
    raise KeyError(f"Pas de grille définie pour {untuned_best_name}")

base_estimators = {
    "LogisticRegression": LogisticRegression(),
    "RandomForestClassifier": RandomForestClassifier(),
    "GradientBoostingClassifier": GradientBoostingClassifier(),
}

cv_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    estimator=base_estimators[untuned_best_name],
    param_grid=param_grids[untuned_best_name],
    scoring="f1",
    cv=cv_tune,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)
grid.fit(X_train, y_train)

print(f"\nGridSearchCV 5-fold (scoring=f1) — {untuned_best_name}")
print(f"Meilleurs params : {grid.best_params_}")
print(f"Meilleur F1 CV   : {grid.best_score_:.4f}")

tuned_model = grid.best_estimator_
y_pred_tuned = tuned_model.predict(X_test)
if hasattr(tuned_model, "predict_proba"):
    y_score_tuned = tuned_model.predict_proba(X_test)[:, 1]
else:
    y_score_tuned = tuned_model.decision_function(X_test)

tuned_metrics = {
    "model": f"{untuned_best_name}+GridSearch",
    "accuracy": accuracy_score(y_test, y_pred_tuned),
    "precision": precision_score(y_test, y_pred_tuned, zero_division=0),
    "recall": recall_score(y_test, y_pred_tuned, zero_division=0),
    "f1": f1_score(y_test, y_pred_tuned, zero_division=0),
    "roc_auc": float(roc_auc_score(y_test, y_score_tuned)),
}

print("\nÉvaluation test — modèle tuné :")
print(
    f"  accuracy={tuned_metrics['accuracy']:.4f}  precision={tuned_metrics['precision']:.4f}  "
    f"recall={tuned_metrics['recall']:.4f}  F1={tuned_metrics['f1']:.4f}  "
    f"ROC-AUC={tuned_metrics['roc_auc']:.4f}"
)
print(classification_report(y_test, y_pred_tuned, digits=3, zero_division=0))
print("Matrice de confusion (tuné) :")
print(confusion_matrix(y_test, y_pred_tuned))

# Tableau de comparaison
comparison = pd.DataFrame(
    [
        {
            "variant": "Précédent (bruit 12%, LR défaut)",
            "exception_rate": PREV_REF["exception_rate"],
            "accuracy": PREV_REF["accuracy"],
            "precision": PREV_REF["precision"],
            "recall": PREV_REF["recall"],
            "f1": PREV_REF["f1"],
            "roc_auc": PREV_REF["roc_auc"],
        },
        {
            "variant": f"Actuel bruit 6% — {untuned_best_name} défaut",
            "exception_rate": EXCEPTION_RATE if "EXCEPTION_RATE" in dir() else 0.06,
            "accuracy": untuned_row["accuracy"],
            "precision": untuned_row["precision"],
            "recall": untuned_row["recall"],
            "f1": untuned_row["f1"],
            "roc_auc": untuned_row["roc_auc"],
        },
        {
            "variant": f"Actuel bruit 6% — {untuned_best_name} GridSearch",
            "exception_rate": EXCEPTION_RATE if "EXCEPTION_RATE" in dir() else 0.06,
            "accuracy": tuned_metrics["accuracy"],
            "precision": tuned_metrics["precision"],
            "recall": tuned_metrics["recall"],
            "f1": tuned_metrics["f1"],
            "roc_auc": tuned_metrics["roc_auc"],
        },
    ]
)

print("\nComparaison (test set) :")
display(
    comparison.style.format(
        {
            "exception_rate": "{:.0%}",
            "accuracy": "{:.4f}",
            "precision": "{:.4f}",
            "recall": "{:.4f}",
            "f1": "{:.4f}",
            "roc_auc": "{:.4f}",
        }
    )
)

delta_vs_untuned = tuned_metrics["f1"] - float(untuned_row["f1"])
delta_vs_prev = tuned_metrics["f1"] - PREV_REF["f1"]
print(f"\nΔ F1 tuné vs défaut (même bruit 6%) : {delta_vs_untuned:+.4f}")
print(f"Δ F1 tuné vs précédent (bruit 12%)  : {delta_vs_prev:+.4f}")

# Retenir le tuné s'il améliore le F1, sinon garder le défaut
if tuned_metrics["f1"] >= float(untuned_row["f1"]):
    eval_models[untuned_best_name] = tuned_model
    # Met à jour test_summary pour la sélection aval
    test_summary = pd.concat(
        [test_summary[test_summary["model"] != untuned_best_name], pd.DataFrame([{
            "model": untuned_best_name,
            "accuracy": tuned_metrics["accuracy"],
            "precision": tuned_metrics["precision"],
            "recall": tuned_metrics["recall"],
            "f1": tuned_metrics["f1"],
            "roc_auc": tuned_metrics["roc_auc"],
        }])],
        ignore_index=True,
    ).sort_values("f1", ascending=False)
    print(f"\n→ Artifacts : {untuned_best_name} remplacé par la version GridSearch.")
else:
    print(f"\n→ Artifacts : on conserve {untuned_best_name} (défaut), F1 tuné non supérieur.")

best_params = grid.best_params_
grid_best_cv_f1 = float(grid.best_score_)
tuning_comparison = comparison.copy()


## Choix du meilleur modèle (F1-score test)

On sélectionne le modèle avec le **meilleur F1-score** sur le jeu de test, car :
- le F1 équilibre **precision** et **recall**, ce qui est plus pertinent que l’accuracy seule quand `IsOnTime` est déséquilibré ;
- l’accuracy de la baseline (`DummyClassifier most_frequent`) peut être trompeusement élevée en classant toujours la classe majoritaire ;
- le F1 test mesure la capacité à détecter correctement les dossiers à temps **sans ignorer** les retards.

Après cette sélection, un **GridSearchCV (5-fold, scoring=F1)** affine les hyperparamètres du meilleur modèle ; la version tunée remplace le modèle retenu si son F1 test est au moins égal.

L’importance des features est ensuite lue via `feature_importances_` (forêts / boosting) ou via la valeur absolue des **coefficients** (régression logistique).


In [ ]:
# Meilleur modèle selon F1 sur le test set (tableau récapitulatif cell précédente)
best_name = test_summary.sort_values("f1", ascending=False).iloc[0]["model"]
best_model = eval_models[best_name]
best_f1 = float(test_summary.loc[test_summary["model"] == best_name, "f1"].iloc[0])

print(f"Meilleur modèle retenu : {best_name}  (F1 test = {best_f1:.4f})")

# Importance : feature_importances_ (tree/boosting) ou |coef_| (logistique)
if hasattr(best_model, "feature_importances_"):
    importances = np.asarray(best_model.feature_importances_, dtype=float)
    importance_label = "feature_importances_"
elif hasattr(best_model, "coef_"):
    coef = np.asarray(best_model.coef_)
    # binaire → shape (1, n_features) ; multi → moyenne des |coef|
    importances = np.abs(coef).ravel() if coef.ndim == 1 else np.abs(coef).mean(axis=0)
    importance_label = "|coefficients|"
else:
    raise AttributeError(f"{best_name} n'expose ni feature_importances_ ni coef_.")

feat_names = list(X.columns)
if len(feat_names) != len(importances):
    feat_names = [f"f_{i}" for i in range(len(importances))]

imp_df = (
    pd.DataFrame({"feature": feat_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

# Top N pour lisibilité si OneHot a beaucoup de colonnes
TOP_N = min(25, len(imp_df))
imp_plot = imp_df.head(TOP_N).iloc[::-1]  # croissant pour barh (haut = plus important)

fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * TOP_N)))
ax.barh(imp_plot["feature"], imp_plot["importance"], color="#000168")
ax.set_xlabel(importance_label)
ax.set_ylabel("Feature")
ax.set_title(f"Importance des features — {best_name} (top {TOP_N}, tri décroissant)")
plt.tight_layout()
plt.show()

print("\nTop features :")
display(imp_df.head(TOP_N))


In [ ]:
from pathlib import Path
import json
import joblib

# Dossier d'artifacts (à côté du notebook)
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Composants de preprocessing issus du ColumnTransformer
num_pipe = preprocessor.named_transformers_["num"]
cat_pipe = preprocessor.named_transformers_["cat"]

scaler = num_pipe.named_steps["scaler"]          # StandardScaler
encoder = cat_pipe.named_steps["onehot"]        # OneHotEncoder
num_imputer = num_pipe.named_steps["imputer"]   # utile en prod avec le scaler
cat_imputer = cat_pipe.named_steps["imputer"]

model_path = ARTIFACTS_DIR / "model.pkl"
encoder_path = ARTIFACTS_DIR / "encoder.pkl"
scaler_path = ARTIFACTS_DIR / "scaler.pkl"
preprocessor_path = ARTIFACTS_DIR / "preprocessor.pkl"
meta_path = ARTIFACTS_DIR / "feature_meta.json"

joblib.dump(best_model, model_path)
joblib.dump(encoder, encoder_path)
joblib.dump(scaler, scaler_path)
# Pipeline complet (imputers + encoder + scaler) pour inférence production
joblib.dump(preprocessor, preprocessor_path)

meta = {
    "best_model": best_name,
    "best_f1_test": best_f1,
    "target": "IsOnTime",
    "numeric_features": list(num_cols),
    "categorical_features": list(cat_cols),
    "feature_names_out": list(X.columns),
    "files": {
        "model": str(model_path),
        "encoder": str(encoder_path),
        "scaler": str(scaler_path),
        "preprocessor": str(preprocessor_path),
    },
}
meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

# Sauvegarde aussi les imputers (nécessaires avant encoder/scaler sur données neuves)
joblib.dump(num_imputer, ARTIFACTS_DIR / "num_imputer.pkl")
joblib.dump(cat_imputer, ARTIFACTS_DIR / "cat_imputer.pkl")

print("Artifacts sauvegardés :")
for p in sorted(ARTIFACTS_DIR.glob("*")):
    print(f"  - {p}  ({p.stat().st_size:,} octets)")

# Smoke-test : reload
loaded_model = joblib.load(model_path)
loaded_pre = joblib.load(preprocessor_path)
_ = loaded_model.predict(loaded_pre.transform(X_raw.head(3)))
print("\nReload OK — predict sur 3 lignes de X_raw via preprocessor + model.")
